# Lecture 7: PGMs, Inference and Learning via Variational Inference — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise the ideas of Lecture 7: why the evidence $p(x)$ is intractable, forward vs reverse KL, the ELBO identity $\log p(x) = \mathcal L(q) + KL(q\parallel p(z|x))$, the mean-field assumption, the score-function (BBVI) and reparameterisation gradient estimators, and the self-study's Bayesian linear regression with a structured guide $q(w_0)\,q(w_1|w_0)$. Everything is done in plain NumPy/SciPy instead of Pyro, so you see every piece the PPL normally hides.

**18 exercises** · 🧠 Concept ×5 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import stats, integrate
rng = np.random.default_rng(7)

# Toy conjugate model used throughout:  z ~ N(0,1),  x | z ~ N(z,1),  observed x = 2
x_obs = 2.0
def log_joint(z, x=x_obs):
    return stats.norm.logpdf(z, 0, 1) + stats.norm.logpdf(x, z, 1)

# Self-study style data for Bayesian linear regression (true w0 = 1.0, w1 = 0.5, noise sd 0.5)
N_blr = 50
x_blr = rng.uniform(0, 5, N_blr)
y_blr = 1.0 + 0.5 * x_blr + 0.5 * rng.normal(size=N_blr)

## Part A · Concepts

### Exercise 1 · Why is the posterior intractable?
*🧠 Concept · ★☆☆*

In Bayes' rule $p(z|x) = \frac{p(x|z)p(z)}{\int p(x,z)\,dz}$, which term causes the trouble, and in which two situations
(named in the lecture) does it become intractable? Then contrast the two families of approximate inference:
MCMC and Variational Inference. What does each one return, and how does VI turn inference into *optimisation*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the denominator. MCMC produces something you can histogram; VI produces a member of a family $\mathcal Q$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The **evidence** (marginal likelihood) $p(x) = \int p(x,z)\,dz$. It is intractable when $z$ is **high-dimensional**
(the integral is over a huge space, so a grid is impossible) or when prior and likelihood are **non-conjugate**
(no closed form).

* **MCMC** draws (correlated) samples $z^{(1)},\dots,z^{(S)}$ whose distribution converges to $p(z|x)$. It is asymptotically exact but slow,
  and convergence is hard to diagnose.
* **VI** picks a tractable family $\mathcal Q = \{q_\lambda\}$ and solves $\hat\lambda = \arg\min_\lambda KL(q_\lambda(z)\parallel p(z|x))$,
  which is a deterministic optimisation problem (solved with (stochastic) gradient ascent on the ELBO). It is fast and scales,
  but is biased: the answer can never be better than the best member of $\mathcal Q$.

</details>

### Exercise 2 · Forward vs reverse KL
*🧠 Concept · ★☆☆*

The true posterior is bimodal (two well separated bumps). You fit a single Gaussian $q$ by

(a) minimising $KL(p\parallel q)$ (forward), (b) minimising $KL(q\parallel p)$ (reverse).

Sketch / describe the solution in each case and name the behaviour ("mean-seeking"/"mode-seeking", "zero-avoiding"/"zero-forcing").
Which one does VI use, and why does that choice make VI **underestimate variance**?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

In $KL(f\parallel g) = \mathbb E_f[\log f/g]$, the expectation is under the *first* argument. Where does the ratio blow up?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Forward KL** $\mathbb E_p[\log p/q]$ blows up wherever $p>0$ but $q\approx 0$, so $q$ must cover *all* mass of $p$:
it becomes a broad Gaussian sitting between the modes (**mean-seeking, zero-avoiding**). For a Gaussian $q$ the optimum is moment matching.

(b) **Reverse KL** $\mathbb E_q[\log q/p]$ blows up wherever $q>0$ but $p\approx0$, so $q$ avoids the gap between the modes and
locks onto **one** mode (**mode-seeking, zero-forcing**).

VI uses the reverse KL because it is an expectation under $q$, which we can sample, and because it leads to the ELBO, which needs only the
un-normalised joint $p(x,z)$. The forward KL needs samples from $p(z|x)$, which is the very thing we can't get.
Penalising mass outside the support of $p$ makes $q$ too narrow, so VI under-reports uncertainty (Critical Insight 4.3).

</details>

### Exercise 3 · Mean field vs structured guide
*🧠 Concept · ★★☆*

In the self-study the prior has the dependency $w_1 \sim \mathcal N(w_0, 10)$ and the posterior of $(w_0,w_1)$ is strongly correlated.

1. What does the mean-field family $q(w_0,w_1)=q(w_0)q(w_1)$ assume, and what does the resulting approximation look like
   relative to a tilted elliptical posterior?
2. The custom guide uses `w1_dynamic_mean = w1_base + w1_coeff * w0`. Which factorisation does this implement, and which posteriors can it
   represent exactly?
3. At convergence the trace shows `coeff: -0.32`. What does the sign tell you?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A product of independent 1-D Gaussians has an axis-aligned covariance. A Gaussian whose mean is linear in $w_0$ gives a joint Gaussian with non-zero covariance.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Mean field assumes the latent variables are **independent** under $q$. The approximation is an axis-aligned ellipse, which can't tilt,
   so it discards the correlation and (under reverse KL) shrinks to fit *inside* the tilted posterior, underestimating marginal variances.
2. $q(w_0,w_1) = q(w_0)\,q(w_1\mid w_0)$ with $q(w_1|w_0) = \mathcal N(b + c\,w_0, s^2)$. If $q(w_0)$ is Gaussian this is a **full
   bivariate Gaussian**, with $\operatorname{Cov}(w_0,w_1) = c\,\sigma_{w_0}^2$. It can therefore represent *any* bivariate Gaussian exactly,
   including the true posterior of Bayesian linear regression (Exercise 16).
3. $c<0$ means $w_0$ and $w_1$ are **negatively correlated** a posteriori: with $x>0$, a higher intercept must be compensated by a lower slope
   for the line to still pass through the data cloud.

</details>

### Exercise 4 · Score function vs pathwise gradients
*🧠 Concept · ★★☆*

Compare the two ELBO-gradient estimators from Critical Insight 4.2 (`Trace_ELBO`-style score function vs reparameterisation).
For each, state (i) what it needs from the model, (ii) whether it works for discrete latent variables, (iii) its variance.
Why did the self-study use a tiny learning rate `lr = 0.0001`, and what does `constraint=constraints.positive` protect against?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The score function never differentiates *through* the sample $\tilde z$; the pathwise estimator writes $z = g_\lambda(\epsilon)$ and does.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | Score function (BBVI, REINFORCE) | Pathwise / reparameterisation |
|---|---|---|
| needs | $\log p(x,z)$ values and $\nabla_\lambda \log q_\lambda$ | $\nabla_z \log p(x,z)$ and a differentiable map $z=g_\lambda(\epsilon)$ |
| discrete $z$ | ✔ works | ✘ (no gradient through a discrete sample) |
| variance | high: the noisy scalar $\log p - \log q$ multiplies the score | low: uses the slope of the log-joint directly |

High-variance gradient estimates mean single steps can be huge and noisy, so a small learning rate is needed to keep SGD stable.
The positivity constraint optimises an unconstrained parameter (e.g. $\log\sigma$ or a softplus) and maps it to $\sigma>0$, so a gradient
step can never produce an invalid negative/zero standard deviation that would crash `Normal(mu, sigma)`.

</details>

### Exercise 5 · Point estimates vs a posterior
*🧠 Concept · ★☆☆*

Your colleague says: "SVI and MAP both end up with a number for $w_1$, so why bother with the variational machinery?"
Give two concrete things the variational posterior $q_\lambda(w)$ gives you that the MAP estimate $\hat w$ does not,
and one way in which the VI uncertainty can still be misleading.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about credible intervals, predictive distributions, and the direction of the KL.

</details>

<details>
<summary><b>✅ Solution</b></summary>

* **Uncertainty estimates**: credible intervals for each parameter (e.g. $w_1 = 0.47 \pm 0.065$ in the self-study trace), and the
  shrinking of epistemic uncertainty as $N$ grows.
* **Posterior predictive** $p(y^*|x^*) = \int p(y^*|x^*,w)q(w)\,dw$: error bars on predictions that widen away from the data,
  useful for decisions and active learning.
* Also: correlations between parameters (with a structured guide).

Caveat: reverse-KL VI is mode-seeking and (with mean field) **underestimates variance**, so the intervals are typically too narrow.

</details>

## Part B · By hand

### Exercise 6 · Factorise the self-study model
*✍️ By hand · ★☆☆*

Write the joint of the Pyro model `lin_reg_model` as a product of local conditionals (PGM factorisation) and evaluate the
**log prior** $\log p(w_0) + \log p(w_1|w_0)$ at $w_0 = 1,\ w_1 = 0.5$ by hand. Recall `dist.Normal(loc, scale)` takes a *standard
deviation*, so $w_0\sim\mathcal N(0, 10^2)$ and $w_1|w_0 \sim \mathcal N(w_0, 10^2)$.

In [ ]:
log_prior_hand = None

check('log prior', log_prior_hand, stats.norm.logpdf(1.0, 0, 10) + stats.norm.logpdf(0.5, 1.0, 10), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\log \mathcal N(v; \mu, \sigma^2) = -\log\sigma - \tfrac12\log 2\pi - \frac{(v-\mu)^2}{2\sigma^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(y, w_0, w_1|x) = p(w_0)\,p(w_1|w_0)\prod_{i=1}^N p(y_i|x_i,w_0,w_1)$.

$\log p(w_0{=}1) = -\log 10 - 0.9189 - \frac{1}{200} = -2.3026 - 0.9189 - 0.005 = -3.2265$

$\log p(w_1{=}0.5|w_0{=}1) = -2.3026 - 0.9189 - \frac{0.25}{200} = -3.2227$

Sum: $\approx -6.4492$. (The prior is almost flat at this scale; the likelihood will dominate.)

```python
log_prior_hand = -6.4492
```

</details>

### Exercise 7 · KL between two Gaussians
*✍️ By hand · ★★☆*

Derive $KL\big(\mathcal N(\mu_1,\sigma_1^2)\parallel \mathcal N(\mu_2,\sigma_2^2)\big) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac12$.
Then compute $KL(\mathcal N(1,1)\parallel\mathcal N(0,4))$ and $KL(\mathcal N(0,4)\parallel\mathcal N(1,1))$ (second argument is a *variance*).
What does the result say about symmetry?

In [ ]:
kl_fwd = None   # KL(N(1,1) || N(0,4))
kl_rev = None   # KL(N(0,4) || N(1,1))

_f = lambda z: stats.norm.pdf(z, 1, 1) * (stats.norm.logpdf(z, 1, 1) - stats.norm.logpdf(z, 0, 2))
_g = lambda z: stats.norm.pdf(z, 0, 2) * (stats.norm.logpdf(z, 0, 2) - stats.norm.logpdf(z, 1, 1))
check('KL(N(1,1)||N(0,4))', kl_fwd, integrate.quad(_f, -30, 30)[0], tol=1e-3)
check('KL(N(0,4)||N(1,1))', kl_rev, integrate.quad(_g, -30, 30)[0], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Write $\log\frac{f}{g}$ for two Gaussian densities and take the expectation under $f$: you need $\mathbb E_f[(z-\mu_2)^2] = \sigma_1^2 + (\mu_1-\mu_2)^2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

First: $\sigma_1=1,\sigma_2=2$, so $\log 2 + 2/8 - 1/2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log\frac{f(z)}{g(z)} = \log\frac{\sigma_2}{\sigma_1} - \frac{(z-\mu_1)^2}{2\sigma_1^2} + \frac{(z-\mu_2)^2}{2\sigma_2^2}$. Under $f$:
$\mathbb E[(z-\mu_1)^2] = \sigma_1^2$ and $\mathbb E[(z-\mu_2)^2] = \sigma_1^2 + (\mu_1-\mu_2)^2$, which gives the formula.

* $KL(\mathcal N(1,1)\parallel\mathcal N(0,4)) = \log 2 + \frac{1+1}{8} - \frac12 = 0.6931 + 0.25 - 0.5 = 0.4431$
* $KL(\mathcal N(0,4)\parallel\mathcal N(1,1)) = \log\frac12 + \frac{4+1}{2} - \frac12 = -0.6931 + 2.5 - 0.5 = 1.3069$

$KL$ is **not symmetric**. Putting the wide distribution first is penalised more: it puts mass where the narrow one has almost none.

```python
kl_fwd = np.log(2) + 2/8 - 0.5
kl_rev = np.log(0.5) + 5/2 - 0.5
```

</details>

### Exercise 8 · Derive the ELBO identity
*✍️ By hand · ★★☆*

Starting from $KL(q(z)\parallel p(z|x)) = \mathbb E_q\left[\log\frac{q(z)}{p(z|x)}\right]$, show
$$\log p(x) = \mathcal L(q) + KL(q(z)\parallel p(z|x)),\qquad \mathcal L(q) = \mathbb E_q\left[\log \frac{p(z,x)}{q(z)}\right],$$
and then that $\mathcal L(q) = \mathbb E_q[\log p(x|z)] - KL(q(z)\parallel p(z))$. Why does maximising $\mathcal L$ over $q$ minimise
the KL to the posterior, even though we never evaluate $p(x)$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Substitute $p(z|x) = p(z,x)/p(x)$ and use that $\log p(x)$ does not depend on $z$. For the second form write $p(z,x) = p(x|z)p(z)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$KL = \mathbb E_q\left[\log\frac{q(z)\,p(x)}{p(z,x)}\right] = \log p(x) - \mathbb E_q\left[\log\frac{p(z,x)}{q(z)}\right] = \log p(x) - \mathcal L(q)$.
Rearranging gives the identity. Since $KL\ge 0$ (Gibbs), $\mathcal L(q) \le \log p(x)$: a **lower bound on the evidence**.

With $p(z,x) = p(x|z)p(z)$: $\mathcal L = \mathbb E_q[\log p(x|z)] + \mathbb E_q\left[\log\frac{p(z)}{q(z)}\right]
= \underbrace{\mathbb E_q[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{KL(q\parallel p(z))}_{\text{regulariser}}$.

$\log p(x)$ is a constant w.r.t. $q$, so every unit that $\mathcal L$ increases is exactly one unit less KL. The ELBO needs only the
joint $p(z,x)$, never the normaliser.

</details>

### Exercise 9 · ELBO of a conjugate toy model
*✍️ By hand · ★★☆*

Model: $z\sim\mathcal N(0,1)$, $x|z\sim\mathcal N(z,1)$, observed $x=2$.

1. Show that the exact posterior is $\mathcal N(1, \tfrac12)$ and that $p(x) = \mathcal N(x; 0, 2)$. Compute $\log p(x{=}2)$.
2. For $q = \mathcal N(m, s^2)$ derive $\mathcal L(m,s) = -\log 2\pi - \tfrac12\big[(2-m)^2 + s^2\big] - \tfrac12\big[m^2+s^2\big] + \tfrac12\log(2\pi e s^2)$.
3. Evaluate the ELBO and the KL gap for $q=\mathcal N(0,1)$ (the prior).

In [ ]:
log_evidence = None   # log p(x=2)
elbo_prior_q = None   # ELBO at m=0, s=1
kl_gap = None         # log p(x) - ELBO

_ev = np.log(integrate.quad(lambda z: np.exp(log_joint(z)), -20, 20)[0])
_el = integrate.quad(lambda z: stats.norm.pdf(z) * (log_joint(z) - stats.norm.logpdf(z)), -20, 20)[0]
check('log evidence', log_evidence, _ev, tol=1e-3)
check('ELBO at q = N(0,1)', elbo_prior_q, _el, tol=1e-3)
check('KL gap', kl_gap, _ev - _el, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Gaussian conjugacy: posterior precision = prior precision + likelihood precision $= 2$; posterior mean = $\frac{1\cdot 2}{2} = 1$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\mathbb E_q[(2-z)^2] = (2-m)^2 + s^2$ and the Gaussian entropy is $\tfrac12\log(2\pi e s^2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Precision $1+1 = 2$, mean $= x/2 = 1$, so $p(z|x) = \mathcal N(1, 1/2)$. Marginally $x = z + \varepsilon$ with independent unit Gaussians,
   so $x\sim\mathcal N(0,2)$ and $\log p(2) = -\tfrac12\log(4\pi) - \tfrac{4}{4} = -1.2655 - 1 = -2.2655$.
2. $\mathcal L = \mathbb E_q[\log p(x|z)] + \mathbb E_q[\log p(z)] + H[q]$, with
   $\mathbb E_q[\log p(x|z)] = -\tfrac12\log 2\pi - \tfrac12((2-m)^2+s^2)$, $\mathbb E_q[\log p(z)] = -\tfrac12\log2\pi - \tfrac12(m^2+s^2)$,
   $H[q] = \tfrac12\log(2\pi e s^2)$.
3. $m=0,s=1$: $\mathcal L = -1.8379 - 2.5 - 0.5 + 1.4189 = -3.4189$. Gap $= -2.2655 + 3.4189 = 1.1534$,
   which equals $KL(\mathcal N(0,1)\parallel\mathcal N(1,\tfrac12)) = \log\sqrt{0.5} + \frac{1+1}{1} - \tfrac12 = 1.1534$ ✔.
   Plugging $m=1, s^2=\tfrac12$ gives $\mathcal L = \log p(x)$: the bound is tight at the true posterior.

```python
log_evidence = -0.5 * np.log(4 * np.pi) - 1.0
elbo_prior_q = -np.log(2 * np.pi) - 0.5 * (4 + 1) - 0.5 * (0 + 1) + 0.5 * np.log(2 * np.pi * np.e)
kl_gap = log_evidence - elbo_prior_q
print(log_evidence, elbo_prior_q, kl_gap)
```

</details>

### Exercise 10 · The score function identity
*✍️ By hand · ★☆☆*

For $q_\lambda(z) = \mathcal N(z; \mu, \sigma^2)$ with $\lambda = (\mu,\sigma)$:

1. Compute $\nabla_\mu \log q$ and $\nabla_\sigma \log q$.
2. Show $\mathbb E_q[\nabla_\lambda \log q_\lambda(z)] = 0$ in general (this is the step that kills the second integral in the BBVI derivation),
   and verify it for both components of the Gaussian.
3. Evaluate both score components at $z = 1.5$, $\mu = 1$, $\sigma = 0.5$.

In [ ]:
score_mu = None
score_sigma = None

_eps = 1e-6
check('d/dmu log q', score_mu, (stats.norm.logpdf(1.5, 1+_eps, 0.5) - stats.norm.logpdf(1.5, 1-_eps, 0.5)) / (2*_eps), tol=1e-4)
check('d/dsigma log q', score_sigma, (stats.norm.logpdf(1.5, 1, 0.5+_eps) - stats.norm.logpdf(1.5, 1, 0.5-_eps)) / (2*_eps), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\log q = -\log\sigma - \tfrac12\log 2\pi - \frac{(z-\mu)^2}{2\sigma^2}$. For 2., use $\mathbb E_q[\nabla\log q] = \int \nabla q = \nabla\int q = \nabla 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\nabla_\mu \log q = \frac{z-\mu}{\sigma^2}$, $\nabla_\sigma \log q = -\frac1\sigma + \frac{(z-\mu)^2}{\sigma^3}$.
2. $\mathbb E_q[\nabla\log q] = \int q\,\frac{\nabla q}{q} = \nabla\int q = \nabla 1 = 0$. Check: $\mathbb E[z-\mu] = 0$ and
   $\mathbb E[(z-\mu)^2]/\sigma^3 - 1/\sigma = \sigma^2/\sigma^3 - 1/\sigma = 0$ ✔.
3. $\nabla_\mu = 0.5/0.25 = 2$, $\nabla_\sigma = -2 + 0.25/0.125 = 0$.

```python
score_mu = 2.0
score_sigma = 0.0
```

</details>

### Exercise 11 · Mean field on a correlated Gaussian
*✍️ By hand · ★★★*

Let the target be $p(z) = \mathcal N(0, \Sigma)$ with $\Sigma = \begin{pmatrix}1&\rho\\ \rho&1\end{pmatrix}$, and $q(z) = q_1(z_1)q_2(z_2)$ Gaussian.
It is a classical result that the reverse-KL optimum has $q_i = \mathcal N(0, 1/\Lambda_{ii})$ where $\Lambda = \Sigma^{-1}$, while the
forward-KL optimum matches the marginals, $q_i = \mathcal N(0, \Sigma_{ii})$.

For $\rho = 0.9$ compute the mean-field (reverse-KL) variance of $z_1$ and the true marginal variance. By what factor is the
standard deviation underestimated?

In [ ]:
mf_var = None      # variance of q_1 under reverse KL
sd_ratio = None    # true marginal sd / mean-field sd

_S = np.array([[1, .9], [.9, 1]])
check('mean-field variance', mf_var, 1 / np.linalg.inv(_S)[0, 0])
check('sd ratio', sd_ratio, np.sqrt(_S[0, 0] * np.linalg.inv(_S)[0, 0]))

<details>
<summary><b>💡 Hint</b></summary>

For a $2\times2$ matrix, $\Lambda_{11} = \frac{\Sigma_{22}}{\det\Sigma} = \frac{1}{1-\rho^2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\Lambda_{11} = 1/(1-\rho^2) = 1/0.19$, so $\operatorname{Var}_q(z_1) = 1-\rho^2 = 0.19$, while the true marginal variance is $1$.
The standard deviation is too small by a factor $1/\sqrt{0.19} \approx 2.29$. The conditional variance $1/\Lambda_{ii}$ is the width of the
posterior *along the axis through the mode*, which is exactly what an axis-aligned $q$ that must avoid zero-density regions can afford.
As $\rho\to1$ the underestimation becomes arbitrarily bad.

```python
mf_var = 1 - 0.9**2
sd_ratio = 1 / np.sqrt(mf_var)
```

</details>

## Part C · Code from scratch

### Exercise 12 · KL: closed form vs Monte Carlo
*💻 Code · ★☆☆*

Implement `kl_gauss(m1, s1, m2, s2)` (arguments are means and **standard deviations**) using the closed form, and
`kl_mc(m1, s1, m2, s2, M)` that estimates $\mathbb E_f[\log f - \log g]$ from $M$ samples of $f$. Compare them for
$f=\mathcal N(1, 1^2)$, $g = \mathcal N(0, 2^2)$ with $M = 200\,000$.

In [ ]:
def kl_gauss(m1, s1, m2, s2):
    # TODO: closed form KL(N(m1,s1^2) || N(m2,s2^2))
    return None

def kl_mc(m1, s1, m2, s2, M=200_000):
    # TODO: sample z ~ N(m1, s1^2) with rng, average log f(z) - log g(z)
    return None

kl_cf = kl_gauss(1, 1, 0, 2)
kl_est = kl_mc(1, 1, 0, 2)

check('closed form', kl_cf, np.log(2) + 2/8 - 0.5)
check('Monte Carlo close to closed form', kl_est, np.log(2) + 2/8 - 0.5, tol=0.02)

<details>
<summary><b>💡 Hint</b></summary>

`z = m1 + s1 * rng.normal(size=M)`, then `stats.norm.logpdf(z, m1, s1) - stats.norm.logpdf(z, m2, s2)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Monte Carlo estimate is unbiased with standard error $\approx \text{sd}(\log f - \log g)/\sqrt M$, a few thousandths here. The same trick is how `Trace_ELBO` estimates the ELBO.

```python
def kl_gauss(m1, s1, m2, s2):
    return np.log(s2 / s1) + (s1**2 + (m1 - m2)**2) / (2 * s2**2) - 0.5

def kl_mc(m1, s1, m2, s2, M=200_000):
    z = m1 + s1 * rng.normal(size=M)
    return np.mean(stats.norm.logpdf(z, m1, s1) - stats.norm.logpdf(z, m2, s2))

kl_cf = kl_gauss(1, 1, 0, 2)
kl_est = kl_mc(1, 1, 0, 2)
print(kl_cf, kl_est)
```

</details>

### Exercise 13 · Monte Carlo ELBO and a grid search
*💻 Code · ★★☆*

For the toy model (`log_joint` in the setup), implement `elbo_mc(m, s, M)` $= \frac1M\sum_j [\log p(\tilde z_j, x) - \log q(\tilde z_j)]$,
$\tilde z_j \sim\mathcal N(m,s^2)$. Use **common random numbers** (the same $\epsilon$ for every $(m,s)$, $\tilde z = m + s\epsilon$)
and evaluate it on a grid $m\in[-1,3]$, $s\in[0.2, 1.5]$. Store the arg-max in `m_best, s_best`. Also confirm that every ELBO value is
below $\log p(x)$.

In [ ]:
eps_crn = rng.normal(size=20_000)

def elbo_mc(m, s, eps=eps_crn):
    # TODO
    return None

m_grid = np.linspace(-1, 3, 81)
s_grid = np.linspace(0.2, 1.5, 66)
m_best, s_best = None, None

check('best m ~ posterior mean 1', m_best, 1.0, tol=0.06)
check('best s ~ posterior sd 0.707', s_best, np.sqrt(0.5), tol=0.06)

<details>
<summary><b>💡 Hint</b></summary>

Loop over the grid (or broadcast), store values in an array `E`, then `np.unravel_index(E.argmax(), E.shape)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The maximiser lands on $m\approx1$, $s\approx 0.71$, i.e. the exact posterior $\mathcal N(1, 1/2)$, because the Gaussian family contains it.
At that point the ELBO equals $\log p(x) = -2.2655$ (up to MC noise) and the KL gap is zero. Common random numbers make the surface smooth so the
grid arg-max is not dominated by sampling noise.

```python
eps_crn = rng.normal(size=20_000)

def elbo_mc(m, s, eps=eps_crn):
    z = m + s * eps
    return np.mean(log_joint(z) - stats.norm.logpdf(z, m, s))

m_grid = np.linspace(-1, 3, 81)
s_grid = np.linspace(0.2, 1.5, 66)
E = np.array([[elbo_mc(m, s) for s in s_grid] for m in m_grid])
i, j = np.unravel_index(E.argmax(), E.shape)
m_best, s_best = m_grid[i], s_grid[j]
print('best', m_best, s_best, 'ELBO', E.max(), '<= log p(x) =', -0.5*np.log(4*np.pi) - 1)
```

</details>

### Exercise 14 · Black Box VI with the score-function estimator
*💻 Code · ★★☆*

Implement BBVI for the toy model with $q = \mathcal N(m, e^{2\ell})$, parameters $\lambda = (m, \ell)$ ($\ell = \log s$ keeps $s>0$,
like `constraints.positive`). Each step:

1. draw $M$ samples $\tilde z_j \sim q_\lambda$,
2. $\hat g = \frac1M\sum_j \big(\log p(\tilde z_j,x) - \log q_\lambda(\tilde z_j)\big)\,\nabla_\lambda\log q_\lambda(\tilde z_j)$,
3. $\lambda \leftarrow \lambda + \eta_t\,\hat g$ with a decaying step $\eta_t = \eta_0/(1 + t/100)$.

Start at $m=0, \ell=0$ and return the final `(m, s)`.

In [ ]:
def bbvi_score(steps=2000, M=200, eta0=0.05, seed=0):
    r = np.random.default_rng(seed)
    m, ell = 0.0, 0.0
    # TODO
    return None

bbvi_result = bbvi_score()

if bbvi_result is not None:
    check('BBVI mean', bbvi_result[0], 1.0, tol=0.1)
    check('BBVI sd', bbvi_result[1], np.sqrt(0.5), tol=0.1)
else:
    check('BBVI', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

With $s = e^\ell$: $\nabla_m\log q = (z-m)/s^2$ and $\nabla_\ell \log q = (z-m)^2/s^2 - 1$ (chain rule on Exercise 10).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Vectorise over the $M$ samples; the weights $f_j = \log p - \log q$ are just a NumPy array.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The estimator only ever evaluates the (un-normalised) log-joint, which is the "black box" part. Because the scalar weight
$\log p - \log q$ fluctuates a lot, we need many samples per step and a decaying (Robbins–Monro) step size; with a constant large step it
jitters around the optimum. A standard trick is to subtract a baseline (control variate) $b$ from the weights, which is unbiased because
$\mathbb E_q[\nabla\log q]=0$ (Exercise 10).

```python
def bbvi_score(steps=2000, M=200, eta0=0.05, seed=0):
    r = np.random.default_rng(seed)
    m, ell = 0.0, 0.0
    for t in range(steps):
        s = np.exp(ell)
        z = m + s * r.normal(size=M)
        f = log_joint(z) - stats.norm.logpdf(z, m, s)
        g_m = np.mean(f * (z - m) / s**2)
        g_l = np.mean(f * ((z - m)**2 / s**2 - 1))
        eta = eta0 / (1 + t / 100)
        m, ell = m + eta * g_m, ell + eta * g_l
    return m, np.exp(ell)

bbvi_result = bbvi_score()
print(bbvi_result)
```

</details>

### Exercise 15 · Gradient variance: score function vs reparameterisation
*💻 Code · ★★☆*

At the fixed point $m = 0, s = 1$ compute $M = 100\,000$ **single-sample** estimates of $\partial\mathcal L/\partial m$ for the toy model with

* the score function: $(\log p(\tilde z,x) - \log q(\tilde z))\,(\tilde z - m)/s^2$,
* the reparameterisation $\tilde z = m + s\epsilon$: $\frac{\partial}{\partial m}[\log p(m+s\epsilon, x) - \log q(m+s\epsilon)] = x - 2\tilde z$
  (the entropy term has no $m$ dependence).

Store the means and variances. Both should be unbiased for the true gradient (what is it, from Exercise 9?). Which has lower variance?

In [ ]:
g_score = None    # array of single-sample estimates
g_reparam = None

if g_score is not None:
    check('score estimator unbiased', np.mean(g_score), 2.0, tol=0.08)
    check('reparam estimator unbiased', np.mean(g_reparam), 2.0, tol=0.05)
    check('reparam has lower variance', np.var(g_reparam) < np.var(g_score), True)
else:
    check('estimators', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

From $\mathcal L(m,s)$ in Exercise 9: $\partial\mathcal L/\partial m = (2-m) - m$, which is $2$ at $m=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

True gradient: $\partial_m\mathcal L = (x-m) - m = 2$. Both estimators average to 2, but the reparameterisation one has variance
$\operatorname{Var}(2 - 2\epsilon) = 4$ while the score function's is about 7 times larger (about 29 here), because it multiplies the noisy
level of $\log p - \log q$ (not its slope) with the score. Lower variance is why Pyro prefers reparameterised (pathwise) gradients for
continuous latents (Critical Insight 4.2).

```python
r = np.random.default_rng(1)
m, s = 0.0, 1.0
eps = r.normal(size=100_000)
z = m + s * eps
g_score = (log_joint(z) - stats.norm.logpdf(z, m, s)) * (z - m) / s**2
g_reparam = x_obs - 2 * z
print('score  : mean %.3f var %.2f' % (g_score.mean(), g_score.var()))
print('reparam: mean %.3f var %.2f' % (g_reparam.mean(), g_reparam.var()))
```

</details>

### Exercise 16 · Exact posterior for the self-study regression
*💻 Code · ★★☆*

The model of `lin_reg_model` is $w_0\sim\mathcal N(0,10^2)$, $w_1|w_0\sim\mathcal N(w_0,10^2)$, $y_i\sim\mathcal N(w_0 + w_1x_i, 1^2)$ (note: the model uses
noise sd 1 even though the data were generated with sd 0.5).

1. Show the prior on $w = (w_0, w_1)$ is jointly Gaussian with mean $0$ and covariance $\Sigma_0 = \begin{pmatrix}100 & 100\\ 100 & 200\end{pmatrix}$.
2. Compute the exact posterior $\mathcal N(\mu_N, \Sigma_N)$ with $\Sigma_N = (\Sigma_0^{-1} + \Phi^\top\Phi)^{-1}$, $\mu_N = \Sigma_N \Phi^\top y$
   (noise variance 1, $\Phi = [\mathbf 1, x]$) on `x_blr, y_blr`. Store `post_mean`, `post_cov` and the correlation `post_corr`.

In [ ]:
Sigma0 = None
post_mean, post_cov, post_corr = None, None, None

# independent reference: brute-force normalisation of prior x likelihood on a 2-D grid
_a = np.linspace(-0.5, 2.5, 301); _b = np.linspace(0.0, 1.0, 301)
_A, _B = np.meshgrid(_a, _b, indexing='ij')
_lp = (stats.norm.logpdf(_A, 0, 10) + stats.norm.logpdf(_B, _A, 10)
       + stats.norm.logpdf(y_blr[None, None, :], _A[..., None] + _B[..., None] * x_blr[None, None, :], 1).sum(-1))
_P = np.exp(_lp - _lp.max()); _P /= _P.sum()
_m = np.array([(_P * _A).sum(), (_P * _B).sum()])
_c01 = (_P * (_A - _m[0]) * (_B - _m[1])).sum()
_C = np.array([[(_P * (_A - _m[0])**2).sum(), _c01], [_c01, (_P * (_B - _m[1])**2).sum()]])
check('posterior mean', post_mean, _m, tol=2e-3)
check('posterior covariance', post_cov, _C, tol=2e-3)
check('posterior correlation', post_corr, _C[0, 1] / np.sqrt(_C[0, 0] * _C[1, 1]), tol=1e-2)

<details>
<summary><b>💡 Hint 1</b></summary>

$w_1 = w_0 + 10\epsilon$, so $\operatorname{Var}(w_1) = 100 + 100$ and $\operatorname{Cov}(w_0,w_1) = \operatorname{Var}(w_0) = 100$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`Phi = np.c_[np.ones(N_blr), x_blr]`; `post_cov = np.linalg.inv(np.linalg.inv(Sigma0) + Phi.T @ Phi)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The posterior mean is close to the true $(1, 0.5)$ and the correlation is strongly negative (about $-0.86$): intercept and slope trade
off, just as the negative `w1_coeff` in the self-study trace suggested. Because everything is linear-Gaussian this posterior is exact.
We use VI here only to *learn* about VI; in non-conjugate models there is no such closed form.

```python
Sigma0 = np.array([[100., 100.], [100., 200.]])
Phi = np.c_[np.ones(N_blr), x_blr]
post_cov = np.linalg.inv(np.linalg.inv(Sigma0) + Phi.T @ Phi)
post_mean = post_cov @ Phi.T @ y_blr
post_corr = post_cov[0, 1] / np.sqrt(post_cov[0, 0] * post_cov[1, 1])
print(post_mean, '\n', post_cov, '\ncorr', post_corr)
```

</details>

### Exercise 17 · The structured guide can be exact
*💻 Code · ★★★*

The self-study guide is $q(w_0) = \mathcal N(\mu_0, \sigma_0^2)$, $q(w_1|w_0) = \mathcal N(\text{base} + \text{coeff}\cdot w_0,\ \text{scale}^2)$.

1. Using the Gaussian conditioning formula, find the guide parameters that make $q$ **equal** to the posterior from Exercise 16. Store them in
   `guide = dict(w0_mean=..., w0_scale=..., w1_base=..., w1_coeff=..., w1_scale=...)`.
2. Sample 200 000 draws from your guide (ancestral sampling: $w_0$ first, then $w_1|w_0$) and confirm the empirical covariance matches `post_cov`.
3. Compare with the **mean-field** reverse-KL optimum, which has variances $1/\Lambda_{ii}$, $\Lambda = \Sigma_N^{-1}$ (Exercise 11). Store
   the mean-field sds in `mf_sd` and the true marginal sds in `true_sd`.

In [ ]:
guide = None
mf_sd, true_sd = None, None

if guide is not None:
    _r = np.random.default_rng(3)
    _w0 = guide['w0_mean'] + guide['w0_scale'] * _r.normal(size=200_000)
    _w1 = guide['w1_base'] + guide['w1_coeff'] * _w0 + guide['w1_scale'] * _r.normal(size=200_000)
    check('guide reproduces posterior covariance', np.cov(_w0, _w1), post_cov, tol=2e-3)
    check('guide reproduces posterior mean', [_w0.mean(), _w1.mean()], post_mean, tol=5e-3)
    check('mean field sds', mf_sd, 1 / np.sqrt(np.diag(np.linalg.inv(post_cov))))
    check('mean field underestimates', np.all(np.asarray(mf_sd) < np.asarray(true_sd)), True)
else:
    check('guide', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

For a bivariate Gaussian: $w_1|w_0 \sim \mathcal N\big(\mu_1 + \frac{\Sigma_{01}}{\Sigma_{00}}(w_0-\mu_0),\ \Sigma_{11} - \frac{\Sigma_{01}^2}{\Sigma_{00}}\big)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

So coeff $=\Sigma_{01}/\Sigma_{00}$, base $=\mu_1 - \text{coeff}\,\mu_0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\text{coeff} = \Sigma_{01}/\Sigma_{00}$ (negative), $\text{base} = \mu_1 - \text{coeff}\cdot\mu_0$, $\text{scale} = \sqrt{\Sigma_{11}-\Sigma_{01}^2/\Sigma_{00}}$,
$\mu_0, \sigma_0$ = posterior mean and sd of $w_0$. With these the ELBO reaches $\log p(y)$ exactly.
The mean-field sds are about half the true marginal sds (correlation $\approx -0.86$ gives a factor $\sqrt{1-\rho^2}\approx 0.5$), which is
the "underestimates parameter uncertainty" picture from slide 9.

```python
S, mu = post_cov, post_mean
coeff = S[0, 1] / S[0, 0]
guide = dict(w0_mean=mu[0], w0_scale=np.sqrt(S[0, 0]),
             w1_base=mu[1] - coeff * mu[0], w1_coeff=coeff,
             w1_scale=np.sqrt(S[1, 1] - S[0, 1]**2 / S[0, 0]))
mf_sd = 1 / np.sqrt(np.diag(np.linalg.inv(S)))
true_sd = np.sqrt(np.diag(S))
print(guide); print('mean-field sd', mf_sd, ' true sd', true_sd)
```

</details>

### Exercise 18 · Mode seeking on a bimodal target
*💻 Code · ★★★*

Target: $p(z) = \tfrac12\mathcal N(-3, 1) + \tfrac12\mathcal N(3, 1)$. Fit $q = \mathcal N(m, s^2)$ by grid search over $m\in[-5,5]$, $s\in[0.3, 4.5]$
(steps 0.05 and 0.02), evaluating both divergences by numerical integration on a fine $z$-grid:

* reverse $KL(q\parallel p)$ → store optimum in `rev_m, rev_s`,
* forward $KL(p\parallel q)$ → store optimum in `fwd_m, fwd_s`.

Interpret the result in terms of Critical Insight 4.3.

In [ ]:
z = np.linspace(-14, 14, 4001); dz = z[1] - z[0]
p_z = 0.5 * stats.norm.pdf(z, -3, 1) + 0.5 * stats.norm.pdf(z, 3, 1)
m_g = np.arange(-5, 5.0001, 0.05)
s_g = np.arange(0.3, 4.5001, 0.02)
rev_m = rev_s = fwd_m = fwd_s = None

if rev_m is not None:
    check('reverse KL locks onto a mode (|m| ~ 3)', abs(rev_m), 3.0, tol=0.15)
    check('forward KL is mean seeking (m ~ 0)', fwd_m, 0.0, tol=0.05)
    check('forward KL matches variance 1 + 9 = 10', fwd_s**2, 10.0, tol=0.2)
else:
    check('KL fits', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Reverse: $\sum_z q(z)[\log q(z) - \log p(z)]\,\Delta z$. Forward: $\sum_z p(z)[\log p(z) - \log q(z)]\,\Delta z$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use `stats.norm.logpdf` for $\log q$ to avoid $\log 0$; mask where $q$ underflows in the reverse KL.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Reverse KL picks $m\approx\pm3$, $s\approx 1$: one mode, ignoring the other entirely (mode-seeking, zero-forcing). Forward KL picks
$m=0$ and $s^2 = \operatorname{Var}_p(z) = 1 + 3^2 = 10$ (moment matching): $q$ covers both modes but puts its peak in the valley where
$p$ has almost no mass (mean-seeking, zero-avoiding). VI uses the reverse KL, so on a multimodal posterior it reports one mode with
over-confident uncertainty.

*Try it:* move the modes to $\pm2$. The reverse-KL optimum then jumps to a single broad Gaussian at $m=0$ ($s\approx2.4$): mode-seeking only kicks in when the valley between the modes is deep enough.

```python
z = np.linspace(-14, 14, 4001); dz = z[1] - z[0]
p_z = 0.5 * stats.norm.pdf(z, -3, 1) + 0.5 * stats.norm.pdf(z, 3, 1)
log_p = np.log(p_z + 1e-300)
m_g = np.arange(-5, 5.0001, 0.05)
s_g = np.arange(0.3, 4.5001, 0.02)
R = np.empty((len(m_g), len(s_g))); F = np.empty_like(R)
for i, m in enumerate(m_g):
    for j, s in enumerate(s_g):
        lq = stats.norm.logpdf(z, m, s); q = np.exp(lq)
        R[i, j] = np.sum(q * (lq - log_p)) * dz
        F[i, j] = np.sum(p_z * (log_p - lq)) * dz
i, j = np.unravel_index(R.argmin(), R.shape); rev_m, rev_s = m_g[i], s_g[j]
i, j = np.unravel_index(F.argmin(), F.shape); fwd_m, fwd_s = m_g[i], s_g[j]
print('reverse KL:', rev_m, rev_s, '| forward KL:', fwd_m, fwd_s)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*